# Script for combining, cleaning, and inspecting Tweede Kamer data

In [1]:
# imports
import pandas as pd
import ast
from IPython.display import display, HTML
import pandas as pd
import re
from html import escape

In [5]:
def deduplicate_block(text):
    if not isinstance(text, str):
        return text
    sentences = text.split(". ")  # splits op zinnen
    unique_sentences = list(dict.fromkeys(sentences))  # behoud volgorde, verwijder dubbels
    return ". ".join(unique_sentences)



In [7]:
beleidsstukken = pd.read_csv(r"C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\Tweede Kamer\beleidsstukken_ai_relevant.csv", index_col=0)
# make it lists
beleidsstukken['company_hits'] = beleidsstukken['company_hits'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
beleidsstukken['matched_keywords_all'] = beleidsstukken['matched_keywords_all'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# combine all keywords found in beleidsstukken in matched_keywords_all
beleidsstukken['matched_keywords_all'] = beleidsstukken.apply(
    lambda row: (row['company_hits'] or []) + (row['matched_keywords_all'] or []),
    axis=1
)

# beleidsstukken = beleidsstukken.reset_index(drop=True)
beleidsstukken.drop(['relevant_word_count','body','text', 'id', 'canonical','matched_keywords_title', 'n_hits_title_total', 'dataurl', 'lastmodified', 'matched_keywords_body'], axis=1, inplace=True)

#reset index
beleidsstukken = beleidsstukken.reset_index(drop=True)

beleidsstukken['type'] = 'beleidsstuk'
beleidsstukken = beleidsstukken.rename(columns={"relevant_text": "body"})



beleidsstukken["body"] = beleidsstukken["body"].apply(deduplicate_block)
# show updates
beleidsstukken.head()

,title,year,ai_related,matched_keywords_all,n_hits_body_total,company_hits,body,word_count,type
0,Investeren in Perspectief (Beleidsnota 2018),2018,yes,"[adyen, google, adyen, google, kunstmatige int...",1,"[adyen, google]",Ook biedt de agenda kansen aan het bedrijfslev...,29318.0,beleidsstuk
1,Nota Defensie Industrie Strategie,2018,yes,"[drones, drones, ai, artificiële intelligentie...",13,[drones],Nederland wil zelf aan militaire kennisontwikk...,25200.0,beleidsstuk
2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021,yes,"[google, google, ai, algoritmes, artificiële i...",12,[google],Zo wordt in de vernieuwde strategie nu ook de ...,4846.0,beleidsstuk
3,Beslisnota's bij de Kamerbrief over toekomstig...,2021,yes,[ai],14,[],25 januari\n37 9-2-2023 Nota-StasGB-Memo box 3...,145567.0,beleidsstuk
4,Beslisnota bij Kamerbrief over aanpak belastin...,2021,yes,[ai],3,[],Dit betreft de nota’s in de onderstaande tabel...,16363.0,beleidsstuk


In [8]:
vergaderstukken = pd.read_csv(r"C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\Tweede Kamer\vergaderstukken_ai_relevant.csv", index_col=0)

#reset index
vergaderstukken = vergaderstukken.reset_index(drop=True)

# make it lists
vergaderstukken['company_hits'] = vergaderstukken['company_hits'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
vergaderstukken['matched_keywords_all'] = vergaderstukken['matched_keywords_all'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# combine all keywords found in beleidsstukken in matched_keywords_all
vergaderstukken['matched_keywords_all'] = vergaderstukken.apply(
    lambda row: (row['company_hits'] or []) + (row['matched_keywords_all'] or []),
    axis=1
)


# beleidsstukken = beleidsstukken.reset_index(drop=True)
vergaderstukken.drop(['relevant_word_count','body','text', 'matched_keywords_title', 'n_hits_title_total', 'matched_keywords_body'], axis=1, inplace=True)

vergaderstukken['type'] = 'vergaderstuk'
vergaderstukken = vergaderstukken.rename(columns={"relevant_text": "body"})

vergaderstukken["body"] = vergaderstukken["body"].apply(deduplicate_block)
vergaderstukken.head()


,title,year,ai_related,matched_keywords_all,n_hits_body_total,company_hits,body,word_count,type
0,Geannoteerde besluitenlijst ministerraad 28 me...,2021,yes,[kunstmatige intelligentie],2,[],Conclusies van de coördinatiecommissie d.d. 25...,5180.0,vergaderstuk
1,Agenda ministerraad 4 juni 2021,2021,yes,"[algoritmen, artificiële intelligentie]",2,[],Programma Landelijke Vreemdelingen Voorziening...,1219.0,vergaderstuk
2,Geannoteerde besluitenlijst ministerraad 4 jun...,2021,yes,"[ai, algoritmen, artificiële intelligentie, ku...",6,[],"1 juni 2021,\nnr.22 (Minister van BZ)\nDe conc...",6588.0,vergaderstuk
3,Geannoteerde besluitenlijst ministerraad 29 ok...,2021,yes,[bard],2,[],4. EU-implementatie\na. Wijziging van het Alge...,6316.0,vergaderstuk
4,Geannoteerde besluitenlijst ministerraad 26 no...,2021,yes,"[ai, artificial intelligence]",2,[],Raad Buitenlandse Zaken (Handel) d.d. 29 novem...,5817.0,vergaderstuk


In [9]:
plenaire_verslagen = pd.read_csv(r"C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\scripts\policy_papers\data\plenaire_verslagen\plenaire_verslagen_relevant_sections.csv")

# drop columns
plenaire_verslagen.drop(['relevant_word_count','text', 'matched_keywords_title', 'n_hits_title_total', 'matched_keywords_body'], axis=1, inplace=True)

plenaire_verslagen['type'] = 'plenair_verslag'

plenaire_verslagen = plenaire_verslagen.rename(columns={"filename": "title"} )
plenaire_verslagen = plenaire_verslagen.rename(columns={"relevant_text": "body"})

plenaire_verslagen["year"] = plenaire_verslagen["title"].str.extract(r"(\d{4})")
plenaire_verslagen["body"] = plenaire_verslagen["body"].apply(deduplicate_block)

plenaire_verslagen.head()

,title,ai_related,matched_keywords_all,n_hits_body_total,company_hits,body,word_count,type,year
0,kamerstukken-plenaire_verslagen-detail-2014-20...,yes,['gemini'],1060,[],Het is nu eind januari. Wanneer komt de minist...,16810683.0,plenair_verslag,2014
1,kamerstukken-plenaire_verslagen-detail-2014-20...,yes,"['asml', 'gemini']",185,['asml'],Ik heb daar met de collega-woordvoerder van me...,8707194.0,plenair_verslag,2014
2,kamerstukken-plenaire_verslagen-detail-2015-20...,yes,"['google', 'algoritmen']",274,['google'],Eerder is gesproken over het belang van jonger...,18539294.0,plenair_verslag,2015
3,kamerstukken-plenaire_verslagen-detail-2015-20...,yes,"['apple', 'facebook', 'robot']",44,"['apple', 'facebook']",Wij voeren kritische gesprekken. Het is belang...,9425317.0,plenair_verslag,2015
4,kamerstukken-plenaire_verslagen-detail-2015-20...,yes,"['drones', 'google', 'tesla', 'uber', 'robot']",39,"['drones', 'google', 'tesla', 'uber']",Mijn eerste punt is dat het curriculum van van...,15082880.0,plenair_verslag,2015


In [10]:
# combine dataframes
tweede_kamer_data = pd.concat([beleidsstukken, vergaderstukken, plenaire_verslagen], ignore_index=True)
tweede_kamer_data = tweede_kamer_data.drop(columns=['ai_related'])
#rename  column
tweede_kamer_data = tweede_kamer_data.rename(columns={"word_count": "original_word_count"})
tweede_kamer_data.head()

,title,year,matched_keywords_all,n_hits_body_total,company_hits,body,original_word_count,type
0,Investeren in Perspectief (Beleidsnota 2018),2018,"[adyen, google, adyen, google, kunstmatige int...",1,"[adyen, google]",Ook biedt de agenda kansen aan het bedrijfslev...,29318.0,beleidsstuk
1,Nota Defensie Industrie Strategie,2018,"[drones, drones, ai, artificiële intelligentie...",13,[drones],Nederland wil zelf aan militaire kennisontwikk...,25200.0,beleidsstuk
2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021,"[google, google, ai, algoritmes, artificiële i...",12,[google],Zo wordt in de vernieuwde strategie nu ook de ...,4846.0,beleidsstuk
3,Beslisnota's bij de Kamerbrief over toekomstig...,2021,[ai],14,[],25 januari\n37 9-2-2023 Nota-StasGB-Memo box 3...,145567.0,beleidsstuk
4,Beslisnota bij Kamerbrief over aanpak belastin...,2021,[ai],3,[],Dit betreft de nota’s in de onderstaande tabel...,16363.0,beleidsstuk


In [12]:
tweede_kamer_data.head()

,title,year,matched_keywords_all,n_hits_body_total,company_hits,body,original_word_count,type
0,Investeren in Perspectief (Beleidsnota 2018),2018,"[adyen, google, adyen, google, kunstmatige int...",1,"[adyen, google]",Ook biedt de agenda kansen aan het bedrijfslev...,29318.0,beleidsstuk
1,Nota Defensie Industrie Strategie,2018,"[drones, drones, ai, artificiële intelligentie...",13,[drones],Nederland wil zelf aan militaire kennisontwikk...,25200.0,beleidsstuk
2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021,"[google, google, ai, algoritmes, artificiële i...",12,[google],Zo wordt in de vernieuwde strategie nu ook de ...,4846.0,beleidsstuk
3,Beslisnota's bij de Kamerbrief over toekomstig...,2021,[ai],14,[],25 januari\n37 9-2-2023 Nota-StasGB-Memo box 3...,145567.0,beleidsstuk
4,Beslisnota bij Kamerbrief over aanpak belastin...,2021,[ai],3,[],Dit betreft de nota’s in de onderstaande tabel...,16363.0,beleidsstuk


In [13]:
tweede_kamer_data.to_csv("tweede_kamer_data_with_comp.csv")

In [ ]:
# add column ai_related with value 'yes' to indicate all documents are ai related
tweede_kamer_data['ai_related'] = 'yes'

In [ ]:
# make sure matched_keywords_all is list
for idx, row in tweede_kamer_data.iterrows():
    if isinstance(row['matched_keywords_all'], str):
        try:
            tweede_kamer_data.at[idx, 'matched_keywords_all'] = ast.literal_eval(row['matched_keywords_all'])
        except (ValueError, SyntaxError):
            tweede_kamer_data.at[idx, 'matched_keywords_all'] = []
tweede_kamer_data['matched_keywords_all'].iloc[1]

In [ ]:


def inspect_ai_related(df, num_samples=30, random_state=42):
    """
    Displays random samples with highlighted *matched* keywords.
    Uses df['matched_keywords_all'] per row instead of a global keyword list.
    """

    # --- sanity checks ---
    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if not {'title', 'body'}.issubset(df.columns):
        missing = {'title', 'body'} - set(df.columns)
        raise KeyError(f"Missing required column(s): {missing}")
    if 'matched_keywords_all' not in df.columns:
        raise KeyError("DataFrame must contain 'matched_keywords_all'.")

    # --- sample ---
    n = min(num_samples, len(df))
    samples = df.sample(n=n, random_state=random_state)

    # --- internal highlight function ---
    def highlight_keywords(text, keywords):
        if pd.isna(text):
            return ""
        s = str(text)

        if not isinstance(keywords, (set, list)):
            raise ValueError("Keywords must be a set or list")
        kws = {str(w).strip().lower() for w in keywords if str(w).strip()}
        if not kws:
            return s

        # sort by length to avoid partial overshadowing
        ordered = sorted(kws, key=len, reverse=True)

        # custom boundary: match even inside hyphenated words
        def make_pattern(word):
            return rf'(?<![A-Za-z0-9]){re.escape(word)}(?![A-Za-z0-9])'

        combined = "|".join(make_pattern(w) for w in ordered)
        regex = re.compile(combined, flags=re.IGNORECASE)

        def repl(m):
            kw = m.group(0)
            return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{escape(kw)}</span>'

        return regex.sub(repl, s)

    # --- display samples ---
    for idx, row in samples.iterrows():
        ai_val = row['ai_related']
        title = row['title']
        body  = row['body']

        # --- use the actually matched keywords in this row ---
        matched_keywords = row.get('matched_keywords_all', [])
        if isinstance(matched_keywords, (list, set, tuple)):
            row_terms = {str(x).lower() for x in matched_keywords}
        else:
            row_terms = {str(matched_keywords).lower()} if matched_keywords else set()

        # --- highlight ---
        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body, row_terms)

        # --- header ---
        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if matched_keywords:
            header_html += f'<br><strong>matched keywords:</strong> {matched_keywords}'
        header_html += '</div>'

        # --- display ---
        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} random articles (with row-specific matched keywords highlighted).")
    return samples

In [75]:
# Suppose df is your DataFrame
inspect_ai_related(tweede_kamer_data, num_samples=25, random_state=22)

Displayed 25 random articles (with row-specific matched keywords highlighted).


,title,year,matched_keywords_all,n_hits_body_total,body,original_word_count,type,ai_related
101,Nota naar aanleiding van verslag wetsvoorstel ...,2024,"[facebook, ai, algoritme, algoritmen, artifici...",39,Het opslaan van internetverkeer van veel niets...,45722.0,beleidsstuk,yes
528,kamerstukken-plenaire_verslagen-detail-2024-20...,2024,"[tesla, uber, ai, algoritmes]",4488,U had nog een tweede verzoek. De heer Stultien...,22299258.0,plenair_verslag,yes
127,Beslisnota bij uitstelbrief antwoorden Kamervr...,2024,"[algoritme, algoritmes]",2,TER BESLUITVORMING\nNota actief openbaar\nJa\n...,228.0,beleidsstuk,yes
460,kamerstukken-plenaire_verslagen-detail-2021-20...,2021,"[facebook, algoritme, algoritmes]",1529,Dan kan hij een uurtje gaan kijken. De heer Pe...,7782793.0,plenair_verslag,yes
370,kamerstukken-plenaire_verslagen-detail-2018-20...,2018,[algoritme],105,Heel kort. De heer El Yassini (VVD):Mijn laats...,22332394.0,plenair_verslag,yes
288,Geannoteerde Agenda voor de inzet van Nederlan...,2024,"[ai, artificial intelligence, kunstmatige inte...",3,Voor het Koninkrijk benadrukken deze mondiale ...,3157.0,vergaderstuk,yes
157,Beslisnota bij antwoord op Kamervragen over Op...,2024,"[ai, generatieve ai, openai]",27,Directoraat-generaal\nEconomie en Digitaliseri...,601.0,beleidsstuk,yes
2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021,"[google, ai, algoritmes, artificiële intellige...",12,Zo wordt in de vernieuwde strategie nu ook de ...,4846.0,beleidsstuk,yes
122,Beslisnota bij bij 5e tussenadvies van de wete...,2024,"[google, ai, algoritmes]",2,Andere conceptkerndoelen digitale geletterdhei...,1817.0,beleidsstuk,yes
500,kamerstukken-plenaire_verslagen-detail-2024-20...,2024,"[drones, ai]",590,De voorzitter:Maar nu weer even terug naar de ...,42535904.0,plenair_verslag,yes
